# Project Lab: Four Portfolio Projects, Built and Tested Step by Step

Notebook for **Chapter 15** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: A Command-Line Expense Tracker

In [ ]:
# expense_tracker/core.py (first part)
import csv
from collections import defaultdict
from datetime import date
from pathlib import Path

FIELDS = ["date", "category", "amount", "note"]

class ExpenseError(Exception):
    """A mistake the user can fix (bad date, bad amount)."""

def validate(day: str, category: str, amount: str, note: str = "") -> dict:
    try:
        date.fromisoformat(day)
    except ValueError:
        raise ExpenseError(f"bad date {day!r}, use YYYY-MM-DD") from None
    try:
        value = round(float(amount), 2)
    except ValueError:
        raise ExpenseError(f"bad amount {amount!r}") from None
    if value <= 0:
        raise ExpenseError("amount must be positive")
    if not category.strip():
        raise ExpenseError("category is empty")
    return {"date": day, "category": category.strip().lower(),
            "amount": value, "note": note}

In [ ]:
def load(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["amount"] = float(r["amount"])
    return rows

def append(path: Path, row: dict) -> None:
    is_new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)

def summarise(rows: list[dict], month: str | None = None) -> dict:
    totals: dict[str, float] = defaultdict(float)
    for r in rows:
        if month and not r["date"].startswith(month):
            continue
        totals[r["category"]] += r["amount"]
    return {k: round(v, 2) for k, v in sorted(totals.items())}

In [ ]:
# expense_tracker/cli.py
import argparse, sys
from pathlib import Path
from . import core

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="expenses", description="Track spending.")
    p.add_argument("--file", type=Path, default=Path("expenses.csv"))
    sub = p.add_subparsers(dest="command", required=True)
    add = sub.add_parser("add", help="record one expense")
    add.add_argument("date"); add.add_argument("category"); add.add_argument("amount")
    add.add_argument("--note", default="")
    lst = sub.add_parser("list", help="show all expenses")
    lst.add_argument("--category")
    summ = sub.add_parser("summary", help="totals per category")
    summ.add_argument("--month", help="YYYY-MM")
    return p

def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    try:
        if args.command == "add":
            row = core.validate(args.date, args.category, args.amount, args.note)
            core.append(args.file, row)
            print(f"saved: {row['date']} {row['category']} {row['amount']:.2f}")
        elif args.command == "list":
            for r in core.load(args.file):
                if args.category and r["category"] != args.category.lower():
                    continue
                print(f"{r['date']}  {r['category']:<10} {r['amount']:>8.2f}  {r['note']}")
        else:
            totals = core.summarise(core.load(args.file), args.month)
            for cat, total in totals.items():
                print(f"{cat:<10} {total:>8.2f}")
            print(f"{'TOTAL':<10} {sum(totals.values()):>8.2f}")
    except core.ExpenseError as e:
        print(f"error: {e}", file=sys.stderr)
        return 1
    return 0

In [ ]:
# tests/test_core.py (excerpt)
class ValidateTests(unittest.TestCase):
    def test_bad_date(self):
        with self.assertRaises(core.ExpenseError):
            core.validate("01/03/2025", "food", "5")

class StorageTests(unittest.TestCase):
    def test_round_trip_and_summary(self):
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / "e.csv"
            core.append(p, core.validate("2025-03-01", "food", "100"))
            core.append(p, core.validate("2025-03-02", "food", "50.25"))
            core.append(p, core.validate("2025-04-01", "travel", "300"))
            rows = core.load(p)
            self.assertEqual(core.summarise(rows), {"food": 150.25, "travel": 300.0})
            self.assertEqual(core.summarise(rows, "2025-03"), {"food": 150.25})

## 15.3 Project: A Reusable Data-Cleaning Package

In [ ]:
# src/tidyshop/clean.py (excerpt)
import re
import pandas as pd

def standardise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """' Order ID ' -> 'order_id'."""
    out = df.copy()
    out.columns = [re.sub(r"\W+", "_", str(c).strip().lower()).strip("_")
                   for c in out.columns]
    return out

def parse_dates(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Bad values become NaT, never an exception."""
    out = df.copy()
    out[column] = pd.to_datetime(out[column], errors="coerce", format="%Y-%m-%d")
    return out

def fill_missing(df: pd.DataFrame, column: str, how: str = "median") -> pd.DataFrame:
    if how not in {"median", "mean"}:
        raise ValueError(f"how must be 'median' or 'mean', got {how!r}")
    out = df.copy()
    out[column] = out[column].fillna(getattr(out[column], how)())
    return out

def flag_outliers_iqr(df: pd.DataFrame, column: str, k: float = 1.5) -> pd.DataFrame:
    out = df.copy()
    q1, q3 = out[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    out[f"{column}_outlier"] = ((out[column] < q1 - k * iqr)
                                | (out[column] > q3 + k * iqr))
    return out

In [ ]:
def clean_orders(df):
    report = {"rows_in": len(df)}
    out = standardise_columns(df)
    out = strip_text(out, ["city", "status"])
    out = parse_dates(out, "order_date")
    report["bad_dates"] = int(out["order_date"].isna().sum())
    before = len(out)
    out = drop_duplicate_rows(out, ["order_id"])
    report["duplicates_dropped"] = before - len(out)
    report["missing_amount"] = int(out["amount"].isna().sum())
    out = fill_missing(out, "amount", "median")
    out = flag_outliers_iqr(out, "amount")
    report["outliers_flagged"] = int(out["amount_outlier"].sum())
    report["rows_out"] = len(out)
    return out, report

In [ ]:
import pandas as pd
from tidyshop import clean_orders
raw = pd.DataFrame({
    "Order ID": [1, 2, 2, 3, 4, 5],
    "City": [" Pune", "pune ", "pune ", "DELHI", "Delhi", "Pune"],
    "Status": ["Shipped"] * 6,
    "Order Date": ["2025-01-01", "2025-01-02", "2025-01-02", "bad", "2025-01-04", "2025-01-05"],
    "Amount": [100.0, 120.0, 120.0, None, 110.0, 9000.0],
})
out, rep = clean_orders(raw)
print(rep)
print(out.to_string())

In [ ]:
def test_input_is_not_changed(self):
    df = pd.DataFrame({"A B": [1]})
    standardise_columns(df)
    self.assertEqual(list(df.columns), ["A B"])

def test_outlier_flag(self):
    df = pd.DataFrame({"x": [10, 11, 12, 13, 14, 500]})
    flags = flag_outliers_iqr(df, "x")["x_outlier"].tolist()
    self.assertEqual(flags, [False] * 5 + [True])

$ PYTHONPATH=src python3 -m unittest discover -s tests -v
...
Ran 8 tests in 0.006s

OK


## 15.4 Project: A Small REST API With a Database

In [ ]:
import json, sqlite3
from http.server import BaseHTTPRequestHandler, HTTPServer

SCHEMA = """CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer TEXT NOT NULL,
    amount REAL NOT NULL CHECK (amount > 0)
)"""

def connect(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row      # rows behave like dicts
    conn.execute(SCHEMA)
    return conn

In [ ]:
def make_handler(db_path):
    class Handler(BaseHTTPRequestHandler):
        def _send(self, status, payload=None):
            body = b"" if payload is None else json.dumps(payload).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def _order_id(self):
            parts = self.path.strip("/").split("/")
            if len(parts) == 2 and parts[0] == "orders" and parts[1].isdigit():
                return int(parts[1])
            return None
        # do_GET, do_POST, do_DELETE ...

In [ ]:
        def do_POST(self):
            if self.path.rstrip("/") != "/orders":
                return self._send(404, {"error": "no such route"})
            size = int(self.headers.get("Content-Length", 0))
            try:
                data = json.loads(self.rfile.read(size))
                customer, amount = str(data["customer"]).strip(), float(data["amount"])
                if not customer or amount <= 0:
                    raise ValueError
            except (ValueError, KeyError, TypeError):
                return self._send(422, {"error": "need customer (text) and amount > 0"})
            conn = connect(db_path)
            try:
                with conn:      # commits on success, rolls back on error
                    cur = conn.execute(
                        "INSERT INTO orders (customer, amount) VALUES (?, ?)",
                        (customer, amount))
                return self._send(201, {"id": cur.lastrowid,
                                        "customer": customer, "amount": amount})
            finally:
                conn.close()

In [ ]:
name = "O'Brien"
c.execute(f"insert into o values ('{name}')")      # wrong
c.execute("insert into o values (?)", (name,))     # right

In [ ]:
def setUp(self):
    self.tmp = tempfile.TemporaryDirectory()
    db = str(Path(self.tmp.name) / "t.db")
    self.srv = HTTPServer(("127.0.0.1", 0), make_handler(db))
    threading.Thread(target=self.srv.serve_forever, daemon=True).start()
    self.base = f"http://127.0.0.1:{self.srv.server_address[1]}"

def tearDown(self):
    self.srv.shutdown(); self.srv.server_close(); self.tmp.cleanup()

$ python3 -m unittest
.....
Ran 5 tests in 2.585s

OK


In [ ]:
# main.py  (not run here)
import sqlite3
from fastapi import FastAPI, HTTPException, Response
from pydantic import BaseModel, Field

app = FastAPI()

class OrderIn(BaseModel):
    customer: str = Field(min_length=1)
    amount: float = Field(gt=0)

def db():
    conn = sqlite3.connect("orders.db")
    conn.row_factory = sqlite3.Row
    return conn

@app.post("/orders", status_code=201)
def create(order: OrderIn):
    with db() as conn:
        cur = conn.execute("INSERT INTO orders (customer, amount) VALUES (?, ?)",
                           (order.customer, order.amount))
    return {"id": cur.lastrowid, **order.model_dump()}

@app.get("/orders/{order_id}")
def read(order_id: int):
    row = db().execute("SELECT * FROM orders WHERE id = ?", (order_id,)).fetchone()
    if row is None:
        raise HTTPException(404, "order not found")
    return dict(row)

## 15.5 Project: A Responsible Web Scraper and Dataset Builder

In [ ]:
import urllib.robotparser

ROBOTS = """
User-agent: *
Disallow: /private/
Crawl-delay: 2
"""

def allowed(robots_text, url, agent="study-bot"):
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(robots_text.splitlines())
    return rp.can_fetch(agent, url), rp.crawl_delay(agent)

(True, 2)      # allowed("https://shop.example/products")
(False, 2)     # allowed("https://shop.example/private/admin")


In [ ]:
from html.parser import HTMLParser

class ProductParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.items, self._cur, self._field = [], None, None

    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "div" and a.get("class") == "product":
            self._cur = {"name": "", "price_text": "", "url": ""}
        elif self._cur is not None:
            if tag in ("h2", "span") and a.get("class") in ("name", "price"):
                self._field = "name" if a["class"] == "name" else "price_text"
            elif tag == "a":
                self._cur["url"] = a.get("href", "")

    def handle_data(self, data):
        if self._cur is not None and self._field:
            self._cur[self._field] += data.strip()

    def handle_endtag(self, tag):
        if tag in ("h2", "span"):
            self._field = None
        elif tag == "div" and self._cur is not None:
            self.items.append(self._cur)
            self._cur = None

In [ ]:
def parse_price(text):
    """'Rs. 1,299' -> 1299.0 ; anything without digits -> None."""
    digits = "".join(ch for ch in text if ch.isdigit() or ch == ".").strip(".")
    return float(digits) if digits else None

In [ ]:
p = ProductParser()
p.feed(PAGE)          # PAGE is the HTML string with three product blocks
for it in p.items:
    print(it)

In [ ]:
import time, urllib.request

UA = "study-bot/0.1 (contact: you@example.com)"
DELAY = 2.0          # at least the site's Crawl-delay

def polite_get(url, cache={}):
    if url in cache:
        return cache[url]
    time.sleep(DELAY)
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=10) as r:
        cache[url] = r.read().decode("utf-8", errors="replace")
    return cache[url]

In [ ]:
import requests
from bs4 import BeautifulSoup

html = requests.get(url, headers={"User-Agent": UA}, timeout=10).text
soup = BeautifulSoup(html, "html.parser")
for box in soup.select("div.product"):
    name = box.select_one(".name").get_text(strip=True)
    price = box.select_one(".price").get_text(strip=True)

In [ ]:
class ParseTests(unittest.TestCase):
    def test_three_products(self):
        p = ProductParser(); p.feed(PAGE)
        self.assertEqual(len(p.items), 3)
    def test_prices(self):
        self.assertEqual(parse_price("Rs. 1,299"), 1299.0)
        self.assertIsNone(parse_price("sold out"))
    def test_robots(self):
        self.assertFalse(allowed(ROBOTS, "https://shop.example/private/x")[0])

## 15.6 Project: A Log Analyser Built Test-First

The projects above start from data. This one starts from a failing test, the way professional teams work. The tool, `loglens`, reads log lines such as `ERROR db: timeout` (the format from Chapter 6) and reports how many lines each level has, which service fails most, and how many lines it could not understand. It is small on purpose, so you can see the whole cycle: write a test, watch it fail, write just enough code, repeat.

In [ ]:
import re
from collections import Counter

LINE = re.compile(r"^(?P<level>INFO|WARNING|ERROR) (?P<service>[\w.]+): (?P<msg>.*)$")

def parse_line(line):
    m = LINE.match(line.strip())
    return m.groupdict() if m else None

def analyse(lines):
    levels, errors, bad = Counter(), Counter(), 0
    for line in lines:
        if not line.strip():
            continue
        rec = parse_line(line)
        if rec is None:
            bad += 1
            continue
        levels[rec["level"]] += 1
        if rec["level"] == "ERROR":
            errors[rec["service"]] += 1
    return {"levels": dict(levels), "top_error_service": errors.most_common(1), "bad_lines": bad}

sample = """INFO api: started
ERROR db: timeout
ERROR db: timeout again
WARNING api: slow response

garbage here
ERROR api: 500 returned""".splitlines()
print(parse_line("ERROR db: timeout"))
print(parse_line("garbage"))
print(analyse(sample))

{'level': 'ERROR', 'service': 'db', 'msg': 'timeout'}
None
{'levels': {'INFO': 1, 'ERROR': 3, 'WARNING': 1}, 'top_error_service': [('db', 2)], 'bad_lines': 1}


In [ ]:
import unittest, re
from collections import Counter
LINE = re.compile(r"^(?P<level>INFO|WARNING|ERROR) (?P<service>[\w.]+): (?P<msg>.*)$")
def parse_line(line):
    m = LINE.match(line.strip())
    return m.groupdict() if m else None
def analyse(lines):
    levels, errors, bad = Counter(), Counter(), 0
    for line in lines:
        if not line.strip():
            continue
        rec = parse_line(line)
        if rec is None:
            bad += 1
            continue
        levels[rec["level"]] += 1
        if rec["level"] == "ERROR":
            errors[rec["service"]] += 1
    return {"levels": dict(levels), "top_error_service": errors.most_common(1), "bad_lines": bad}

class AnalyseTests(unittest.TestCase):
    def test_empty_input(self):
        self.assertEqual(analyse([]), {"levels": {}, "top_error_service": [], "bad_lines": 0})
    def test_blank_lines_are_not_bad(self):
        self.assertEqual(analyse(["", "   "])["bad_lines"], 0)
    def test_garbage_is_counted(self):
        self.assertEqual(analyse(["nonsense"])["bad_lines"], 1)
    def test_top_error_service(self):
        lines = ["ERROR db: a", "ERROR db: b", "ERROR api: c"]
        self.assertEqual(analyse(lines)["top_error_service"], [("db", 2)])
    def test_lowercase_level_is_rejected(self):
        self.assertIsNone(parse_line("error db: x"))

import io
suite = unittest.defaultTestLoader.loadTestsFromTestCase(AnalyseTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran", result.testsRun, "ok" if result.wasSuccessful() else "FAILED")

ran 5 ok


In [ ]:
import argparse, io, json, re, sys
from collections import Counter
LINE = re.compile(r"^(?P<level>INFO|WARNING|ERROR) (?P<service>[\w.]+): (?P<msg>.*)$")

def analyse(lines):
    levels, bad = Counter(), 0
    for line in lines:
        if not line.strip():
            continue
        m = LINE.match(line.strip())
        if m is None:
            bad += 1
        else:
            levels[m["level"]] += 1
    return {"levels": dict(levels), "bad_lines": bad}

def main(argv, stdin):
    p = argparse.ArgumentParser(prog="loglens")
    p.add_argument("--max-bad", type=int, default=0, help="allowed unparsable lines")
    args = p.parse_args(argv)
    report = analyse(stdin)
    print(json.dumps(report, sort_keys=True))
    return 0 if report["bad_lines"] <= args.max_bad else 1

fake = io.StringIO("INFO a: ok\nERROR a: boom\nnonsense\n")
print("exit", main([], fake))
fake = io.StringIO("INFO a: ok\nERROR a: boom\nnonsense\n")
print("exit", main(["--max-bad", "1"], fake))

{"bad_lines": 1, "levels": {"ERROR": 1, "INFO": 1}}
exit 1
{"bad_lines": 1, "levels": {"ERROR": 1, "INFO": 1}}
exit 0


## Chapter recap

## Practice set

Problem 15.1 (easy) Write three success criteria for the expense tracker. **Answer:** Any measurable three, for example: `summary --month 2025-03` returns 245.50 for the sample rows; a bad date returns exit code 1 with a message; 6 tests pass.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch15.html